# Map Showcase to StoryMap Pipeline

In [1]:
from arcgis.gis import GIS
gis = GIS(
    url="https://mapit.mapsdevext.arcgis.com/",
    username="wponcy_dev",
    password="Youstayforme2026!"
)
from arcgis.apps.storymap import StoryMap, story, story_content
from arcgis.apps.storymap.story_content import Image, TextStyles, Video, Audio, Embed, Map, Cover, Text, Button, Gallery, Swipe, Sidecar, Timeline, SidecarSlide, Scales
from arcgis.apps.storymap.collection import Collection 
import arcgis.geometry as arcGeo
import time

# Creating StoryMaps from a Showcase's web maps

In [3]:
class webMap:
    def __init__(self, itemId, geometry):
                # creating a union of the geometries which will be the centerpoint of our map 

        # gathering details from the item
        self.itemId = itemId
        self.item = gis.content.get(item_id)
        self.title = self.item.title
        self.thumbnail_path = self.item.download_thumbnail()
        self.thumbnail = Image(self.thumbnail_path)
        self.summary = self.item.snippet
        self.description = self.item.description
        self.geometry = arcGeo.union(all_geometries, spatial_ref = srs)[0] 
        self.geometryExtent = self.geometry.extent

def createStoryMapforWebMap(webMapObject):

    print('Creating the empty StoryMap')
    # creating an empty storymap
    new_story = StoryMap()

    # filling in the header & byline
    
    new_story.contents[0].title = webMapObject.title
    new_story.contents[0].summary = 'Made with the ArcGIS API for Python'
    new_story.contents[0].date = 'current-date'
    

    # adding an image to the storymap
    print('Adding an image')
    new_node = new_story.add(webMapObject.thumbnail, position = 2)
    
    print('Adding a map and setting focus area extent to:', webMapObject.geometry.extent)
    my_map = Map(webMapObject.itemId)
    new_node = new_story.add(my_map, webMapObject.title)
    my_map.set_viewpoint(extent={ # changing the map's focus area
        'spatialReference': {'latestWkid': 3857, 'wkid': 102100},
        'xmin':  webMapObject.geometry.extent[0], # x-min
        'ymin':  webMapObject.geometry.extent[1], # y-min
        'xmax':  webMapObject.geometry.extent[2], # x-max,
        'ymax':   webMapObject.geometry.extent[3], # y-max
    })  

    while True:
        sidecar = input("Would you like to include a sidecar element? Type 'yes' or 'no'.")
        if sidecar.lower() == "yes":
            # adding a sidecar
            heading = Text(text=webMapObject.title, style=TextStyles.HEADING)  
            new_node = new_story.add(heading)
            paragraph = Text(text="this is a test paragraph, included to test the functionality of the API", style=TextStyles.PARAGRAPH)
            new_node = new_story.add(paragraph)

            
            slide = SidecarSlide(content=paragraph, media=my_map)
            sidecar = Sidecar(style="docked")
            sidecar.slides = [slide]
            new_node = new_story.add(sidecar)
            break
        elif sidecar.lower() == "no":
            break
        else:
            print("Please enter 'yes' or 'no' as a response.")
            
    # adding credits
    new_story.credits("StoryMaps" , "Python API", "Thank You for Reading","Esri Living Atlas of the World")
    
    print('Saving the StoryMap')
    storyMapTitle = f'StoryMap for {webMapObject.title}'
    new_story.save(title=storyMapTitle) # saving it but not publishing
    
    print('Updating the item details')
    print(updateStoryMapItemDetails(storyMapTitle, webMapObject.description, webMapObject.thumbnail_path))
    print('.' * 50)

    # returning the StoryMap, so we can use it in a collection
    return new_story
    
def updateStoryMapItemDetails(SMtitle, WMdescription, WMthumbnail, retries=5, delay=5):
    # time.sleep(20)

    for attempt in range(1, retries):
        story_item_search = gis.content.search(query=f'title:"{SMtitle}"',item_type="StoryMap")
        if story_item_search:
            story_item = story_item_search[0]
            try: 
                # then change the description
                update_status = story_item.update(
                    item_properties={
                        "access": "public",
                        "description": str(WMdescription),
                        "snippet": "This StoryMap was made with the ArcGIS API for Python",
                    },
                    thumbnail = WMthumbnail # updating thumbnail
                )

                if update_status:
                    return("Successfully updated the StoryMap.")
                else: 
                    return("Failure updating StoryMap.")
        
            except Exception as e:
                return(f"An unexpected error occurred: {e}")
        else:
            waitTime = attempt * delay
            print(f'No item found with the query "query="title:{SMtitle}", item_type="StoryMap", waiting {waitTime} seconds.')
            time.sleep(waitTime)
    print('.' * 50)
    
class StopExecution(Exception):
    def _render_traceback_(self):
        return []

def createCollectionFromStoryMaps(storyMapList, showcaseTitle):
    coll = Collection()
    coll.title = showcaseTitle # taking title directly from the map showcase
    coll.content[0].title = showcaseTitle # also taking the collection's cover from the map showcase
    coll.content[0].summary = 'Made with the ArcGIS API for Python'
    for sm in all_storymaps_from_showcase:
        print('storymap', sm)
        webmap_id = str(sm).rsplit("/")[-1]
        webmap_item = gis.content.get(webmap_id)
        coll.add(webmap_item)
    collectionTitle = showcaseTitle.replace("Showcase", "StoryMap Collection") 
    coll.save(title=collectionTitle, access='public')

    return coll
    
# try:
showcaseID = input('Paste in an item ID for a Showcase item.')
showcase = gis.content.get(showcaseID)
# except error as e:
    # print(f"Can't get item {showcaseID} with error: {e} ")
    # raise StopExecution

print('=' * 25, 'CREATING INDIVIDUAL STORYMAPS', '=' * 25 )
all_showcase_data = showcase.get_data()

all_storymaps_from_showcase = []
    
for item in all_showcase_data['items']:
    
    if item['type'] == "Web Map":

        item_id = item['id']
        
        # creating a union of the geometries 
        all_geometries = []
        for geometry in item['highlightedArea']['graphics']:
            srs = geometry['geometry']['spatialReference']
            current_geometry = arcGeo.Geometry(geometry['geometry'])
            all_geometries.append(current_geometry)
        union = arcGeo.union(all_geometries, spatial_ref = srs)[0]

        print('Creating a webmap for id', item_id)
        currentWebMap = webMap(item_id, union)
        
        print(f'Creating StoryMap for item id: {currentWebMap.title}')
        
        all_storymaps_from_showcase.append(createStoryMapforWebMap(currentWebMap))
        
print('=' * 25, 'CONVERTING STORYMAPS TO COLLECTION', '=' * 25 )
collection_from_showcase = createCollectionFromStoryMaps(all_storymaps_from_showcase, showcase.title)
collection_from_showcase

Paste in an item ID for a Showcase item. e9b581d8803845dda64ca8fbb0ba8c51


========================= CREATING INDIVIDUAL STORYMAPS =========================
Creating a webmap for id 7d218a4ccee54bf8888d107fbceb8512
Creating StoryMap for item id: Monthly Soil Moisture
Creating the empty StoryMap
Adding an image
Adding a map and setting focus area extent to: (-12918626.6226, 5521048.055100001, -11581647.0004, 6275089.763599999)


Would you like to include a sidecar element? Type 'yes' or 'no'. yes


Saving the StoryMap
Updating the item details
No item found with the query "query="title:StoryMap for Monthly Soil Moisture", item_type="StoryMap", waiting 5 seconds.
Successfully updated the StoryMap.
..................................................
Creating a webmap for id d6fd1e9992fa42c8bcdd40927c4e6f1c
Creating StoryMap for item id: Monthly Snow Pack
Creating the empty StoryMap
Adding an image
Adding a map and setting focus area extent to: (-12918626.6226, 5521048.055100001, -11581647.0004, 6275089.763599999)


Would you like to include a sidecar element? Type 'yes' or 'no'. yes


...

Would you like to include a sidecar element? Type 'yes' or 'no'. yes


Saving the StoryMap
Updating the item details
Successfully updated the StoryMap.
..................................................
Creating a webmap for id 2ac4dd40336d4bcdb3ccaf6b88304e5d
Creating StoryMap for item id: Monthly Evapotranspiration
Creating the empty StoryMap
Adding an image
Adding a map and setting focus area extent to: (-12918626.6226, 5521048.055100001, -11581647.0004, 6275089.763599999)


Would you like to include a sidecar element? Type 'yes' or 'no'. yes


Saving the StoryMap
Updating the item details
Successfully updated the StoryMap.
..................................................
========================= CONVERTING STORYMAPS TO COLLECTION =========================
storymap https://storymaps.arcgis.com/stories/d27e732e4bb14b4283a8e64e0b6c54f0
storymap https://storymaps.arcgis.com/stories/44babf5ae1e64f98bebe68d782ee5f96
storymap https://storymaps.arcgis.com/stories/2f3e878b01f44d1fb9010dc545ba4f99
storymap https://storymaps.arcgis.com/stories/41cfd077b2d84e90827a7c8b72f1953d
storymap https://storymaps.arcgis.com/stories/8f462b3c5dd44ba492c96f22fb76a3f5


Collection(url=https://storymaps.arcgis.com/collections/5243378722dd451a91c12a9d839bbff6/edit)

In [ ]:
showcase = gis.content.get('05873627e24a43aa84b96be621b62faa')
data = showcase.get_data()
data

In [2]:
test = StoryMap()
mapScale = Scales({'scale': 3000000, 'zoom': 7})

# we'll use separate maps for the map element and the in-sidecar map
standalone_map = Map('7d218a4ccee54bf8888d107fbceb8512')
standalone_map.set_viewpoint(extent={ # changing the map's focus area
        'spatialReference': {'latestWkid': 3857, 'wkid': 102100},
        'xmin':  -12918626.6226, # x-min
        'ymin':  5521048.055100001, # y-min
        'xmax':  -11581647.0004, # x-max,
        'ymax':   6275089.763599999, # y-max
    }, 
                             scale=mapScale
)  
new_node = test.add(standalone_map)

sidecar_map = Map('7d218a4ccee54bf8888d107fbceb8512')
sidecar_map.set_viewpoint(extent={ # changing the map's focus area
        'spatialReference': {'latestWkid': 3857, 'wkid': 102100},
        'xmin':  -12918626.6226, # x-min
        'ymin':  5521048.055100001, # y-min
        'xmax':  -11581647.0004, # x-max,
        'ymax':   6275089.763599999, # y-max
    },
                              scale=mapScale
)  
paragraph = Text(text="this is a test paragraph, included to test the functionality of the API", style=TextStyles.PARAGRAPH)
slide = SidecarSlide(content=paragraph, 
                     media=sidecar_map)
sidecar = Sidecar(style="fixed")
sidecar.slides = [slide]
test.add(sidecar)

# saving
test.save(title='StoryMap with SideCar test')
test

https://storymaps.arcgis.com/stories/5f0f8c57b4304aeaac92ea6e89910d75